<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 03 — Entregável Marketing (challenge 004)

Análise + estratégia. Sem `engagement_rate` nativo: `(likes + shares + comments_count) / views`. Inclua Bilibili e RedNote.

1. Rode o **LAB 01** antes. CSV 23 MB no Volume: `/Volumes/workspace/social/dados`.
2. **Run All**. Compare org/ads **controlando** plataforma × categoria × faixa de `follower_count`.
3. `creator_name` não é PK. Sem coluna de custo — ROI é hipótese.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "social")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

### Tile 1 — Volume (não é o insight)

In [ ]:
display(
    spark.sql(
        """
SELECT platform, content_type, content_category, COUNT(*) AS posts
FROM social_media_dataset
GROUP BY platform, content_type, content_category
ORDER BY posts DESC
"""
    )
)

### Tile 2 — Taxa derivada controlada (o que gera engagement)

In [ ]:
display(
    spark.sql(
        """
SELECT
  platform,
  content_type,
  content_category,
  CASE
    WHEN follower_count < 10000 THEN 'lt_10k'
    WHEN follower_count < 50000 THEN '10k_50k'
    WHEN follower_count < 200000 THEN '50k_200k'
    ELSE '200k_plus'
  END AS creator_band,
  is_sponsored,
  COUNT(*) AS posts,
  ROUND(AVG((likes + shares + comments_count) / views), 4) AS eng_per_view,
  ROUND(AVG(views), 0) AS avg_views,
  ROUND(AVG(shares), 1) AS avg_shares
FROM social_media_dataset
GROUP BY platform, content_type, content_category,
  CASE
    WHEN follower_count < 10000 THEN 'lt_10k'
    WHEN follower_count < 50000 THEN '10k_50k'
    WHEN follower_count < 200000 THEN '50k_200k'
    ELSE '200k_plus'
  END,
  is_sponsored
HAVING COUNT(*) >= 30
ORDER BY eng_per_view DESC
"""
    )
)

### Tile 3 — Orgânico vs patrocinado (média crua vs controlada)

Se as duas médias cruas forem ~0,199, declare: o dump é quase constante. Insight não pode ser “patrocínio não funciona” sem o tile 2.

In [ ]:
display(
    spark.sql(
        """
SELECT
  is_sponsored,
  COUNT(*) AS posts,
  ROUND(AVG((likes + shares + comments_count) / views), 4) AS eng_per_view_raw,
  ROUND(stddev((likes + shares + comments_count) / views), 4) AS eng_std,
  ROUND(AVG(views), 0) AS avg_views,
  ROUND(AVG(follower_count), 0) AS avg_followers
FROM social_media_dataset
GROUP BY is_sponsored
"""
    )
)

### Tile 4 — Audiência (bucket, não JSON) e o que não funciona

In [ ]:
display(
    spark.sql(
        """
SELECT
  audience_age_distribution,
  audience_gender_distribution,
  audience_location,
  platform,
  content_category,
  COUNT(*) AS posts,
  ROUND(AVG((likes + shares + comments_count) / views), 4) AS eng_per_view
FROM social_media_dataset
GROUP BY audience_age_distribution, audience_gender_distribution, audience_location,
         platform, content_category
HAVING COUNT(*) >= 50
ORDER BY eng_per_view ASC
LIMIT 20
"""
    )
)

### Tile 5 — Top 15 para concentrar

Mesmo recorte do tile 2, só as 15 células de maior taxa por view. `views` zero sai da média (`NULLIF`). Não é “vídeo > imagem”.

In [ ]:
display(
    spark.sql(
        """
SELECT
  platform,
  content_type,
  content_category,
  CASE
    WHEN follower_count < 10000 THEN 'lt_10k'
    WHEN follower_count < 50000 THEN '10k_50k'
    WHEN follower_count < 200000 THEN '50k_200k'
    ELSE '200k_plus'
  END AS creator_band,
  is_sponsored,
  COUNT(*) AS posts,
  ROUND(AVG((likes + shares + comments_count) / NULLIF(views, 0)), 4) AS eng_per_view,
  ROUND(AVG(shares), 1) AS avg_shares
FROM social_media_dataset
GROUP BY platform, content_type, content_category,
  CASE
    WHEN follower_count < 10000 THEN 'lt_10k'
    WHEN follower_count < 50000 THEN '10k_50k'
    WHEN follower_count < 200000 THEN '50k_200k'
    ELSE '200k_plus'
  END,
  is_sponsored
HAVING COUNT(*) >= 30
ORDER BY eng_per_view DESC
LIMIT 15
"""
    )
)

### Estratégia (Head de Marketing)

Preencha com os números dos tiles — não com “vídeo > imagem”.

| Decisão | Como achar no dashboard |
| --- | --- |
| Onde concentrar | Maior `eng_per_view` **e** `avg_shares` no tile 2 (plataforma × tipo × categoria × faixa) |
| Política de sponsor | Só patrocinar a célula do tile 2 em que `is_sponsored = true` supera o orgânico **no mesmo** recorte; se o tile 3 for ~igual, threshold = não gastar até haver custo real |
| Parar | 20 piores células do tile 4 + células do tile 2 com muitos posts e `eng_per_view` no fundo |
| Quick wins | Esta semana: copiar o top 3 do tile 2 (mesmo `content_type` + `content_category` + `creator_band`) |

Diferencial: Lab 04 (modelo). Este notebook é a decisão recorrente do time.

### Painéis gráficos

Ranking, galeria, classificação, segmentação, segmentação de imagem, estatísticas do modelo, matriz de confusão, curva de aprendizado e viés/variância. Calculados só com colunas reais do dump.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / '03_LAB_Dashboard', HERE.parent / 'lib', HERE.parent.parent / 'lib'):
    if (candidate / 'lab03_panels' / '__init__.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('lab03_panels não encontrado')

from lab03_panels import build_social_panels
from lab03_panels.render import render_panels

try:
    show = display
except NameError:
    show = print

panels = build_social_panels()
render_panels(panels, show)
print('secoes', list(panels.panels))
